<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-02-grafos/%2018_Grafos_de_Logistica_de_Insumos_e_Produtos_Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 — Grafos de logística de insumos e aplicação agrícola

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**

O nome mantém `Produtos_Acabados` para rastreabilidade com a referência. Na planta agrícola, o resultado é **aplicação registrada por missão e talhão**, não estoque de fertilizante ensacado.


## Duas redes para o mesmo sistema

| Rede | Vértices e arestas | Peso | Pergunta |
|---|---|---|---|
| Materiais $G_M$ | Insumos, preparo, estoque de calda, abastecimento, drone, talhão | Metros estimados entre etapas | Como cada insumo chega à aplicação? |
| Apoio terrestre $G_V$ | Portão, depósito, preparo, base e manutenção | Metros de circulação permitida | Como o veículo de apoio retorna ao portão? |

A mesma classe discreta atende às duas redes, mas as arestas não são intercambiáveis. Um veículo terrestre não percorre a conexão aérea DRONE→TALHÃO. Os pesos descrevem deslocamentos entre etapas; não incluem tempo de mistura, inspeção, carga de bateria ou vazão de abastecimento.

![Logística da planta](attachment:Grafo_Logitica.jpeg)


## Fluxo de materiais e precedência

ÁGUA e PRODUTO convergem para PREPARO. Seguem ESTOQUE_CALDA, ABASTECIMENTO, DRONE e TALHÃO. O subgrafo é um DAG, permitindo ordenação topológica pelo algoritmo de Kahn. A ordem demonstra precedência, mas **não verifica estoque nem receita**. O preparo necessita de ambos os insumos; a rota de menor custo de um deles não autoriza omitir o outro.

Da água ao talhão: $12+5+10+3+180=210$ m. Do produto ao talhão: $8+5+10+3+180=206$ m. Esses valores servem para comparar movimentações por origem; não definem composição química, dosagem agronômica ou custo global de produção.

O notebook condiciona um abastecimento de 24 L à disponibilidade de calda, nível de solo pelo menos 20%, drone pousado e capacidade embarcada de 30 L. Um tanque de solo com 250 L passa a 226 L. Ao associar uma missão de aplicação de 5 L, restam 19 L no drone, com área hipotética de 0,5 ha e consumo registrado de 10 L/ha.


## Circulação de apoio e logística reversa

O anel terrestre segue PORTÃO→DEPÓSITO→PREPARO→BASE→MANUTENÇÃO→PORTÃO. O trajeto BASE→MANUTENÇÃO→PORTÃO mede 65 m. O retorno cria um ciclo dirigido, portanto essa rede não admite ordenação topológica.

A troca e recarga de baterias e o encaminhamento de embalagens, água de lavagem e resíduos exigem fluxos próprios. Estão identificados no layout auxiliar como extensão de escopo, sem conectar resíduos de volta à calda limpa. Não se modelam procedimentos químicos de descarte.


## Rastreabilidade SCADA

Cada registro associa lote, missão, talhão, litros abastecidos, aplicados e remanescentes, área e caminho selecionado. A disponibilidade física não comprova mistura pronta; o exemplo exige também `lote_liberado=True`, representando uma confirmação de processo.

Ao bloquear ESTOQUE_CALDA→ABASTECIMENTO, Dijkstra retorna ausência de rota, impedindo planejar a transferência. O estoque permanece cadastrado: uma via indisponível não elimina seu conteúdo do inventário.

**Entregável:** classe `GrafoLogistico`, modelos distintos de materiais e apoio, ordenação topológica, caminhos mínimos, bloqueio e registro de consumo ligados às variáveis da Etapa 1.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [ ]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [ ]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [ ]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [ ]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [ ]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


In [ ]:
def reconstruir(pred, destino):
    if destino not in pred:
        return []
    caminho, u = [], destino
    while u is not None:
        caminho.append(u); u = pred[u]
    return caminho[::-1]


In [ ]:
def bfs(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return [], {}, {}
    pred, nivel, fila = {origem: None}, {origem: 0}, deque([origem])
    while fila:
        u = fila.popleft()
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in pred:
                pred[v], nivel[v] = u, nivel[u]+1
                fila.append(v)
    return reconstruir(pred, destino), pred, nivel


In [ ]:
def dfs_caminhos(g, origem, destino, modo=None, bloqueados=()):
    """Enumera caminhos simples. Custo depende do número de caminhos, não é O(V+E)."""
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return []
    resultado = []
    def visitar(u, caminho):
        if u == destino:
            resultado.append(caminho[:]); return
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in caminho:
                visitar(v, caminho+[v])
    visitar(origem, [origem])
    return resultado


In [ ]:
def dijkstra(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return inf, []
    dist, pred, fila = {origem: 0.0}, {origem: None}, [(0.0, origem)]
    while fila:
        du, u = heappop(fila)
        if du != dist[u]:
            continue
        if u == destino:
            return du, reconstruir(pred, destino)
        for v, peso, _ in g.vizinhos(u, modo, bloqueados):
            if not isfinite(peso) or peso < 0:
                raise ValueError('Dijkstra exige pesos finitos não negativos')
            candidato = du+peso
            if candidato < dist.get(v, inf):
                dist[v], pred[v] = candidato, u
                heappush(fila, (candidato, v))
    return inf, []


In [ ]:
class GrafoLogistico(GrafoTubulacao):
    """Mesma estrutura discreta; peso em metros de movimentação logística."""
    def adicionar_rota(self, id_, origem, destino, metros):
        self.adicionar_tubulacao(id_,origem,destino,metros,modos=('logistica',))


In [ ]:
def criar_logistica():
    materiais=GrafoLogistico(['AGUA','PRODUTO','PREPARO','ESTOQUE_CALDA','ABASTECIMENTO','DRONE','TALHAO'])
    for i,(u,v,w) in enumerate([('AGUA','PREPARO',12),('PRODUTO','PREPARO',8),
             ('PREPARO','ESTOQUE_CALDA',5),('ESTOQUE_CALDA','ABASTECIMENTO',10),
             ('ABASTECIMENTO','DRONE',3),('DRONE','TALHAO',180)]):
        materiais.adicionar_rota(f'm{i}',u,v,w)
    veiculos=GrafoLogistico(['PORTAO','DEPOSITO','PREPARO','BASE','MANUTENCAO'])
    for i,(u,v,w) in enumerate([('PORTAO','DEPOSITO',25),('DEPOSITO','PREPARO',20),
             ('PREPARO','BASE',15),('BASE','MANUTENCAO',30),('MANUTENCAO','PORTAO',35)]):
        veiculos.adicionar_rota(f'v{i}',u,v,w)
    return materiais,veiculos


In [ ]:
def ordem_topologica(g):
    grau = {v:0 for v in g.vertices}
    for u in g.vertices:
        for v,_,_ in g.vizinhos(u): grau[v]+=1
    fila = deque(v for v in g.vertices if grau[v]==0); ordem=[]
    while fila:
        u=fila.popleft(); ordem.append(u)
        for v,_,_ in g.vizinhos(u):
            grau[v]-=1
            if grau[v]==0: fila.append(v)
    if len(ordem)!=len(g.vertices): raise ValueError('Grafo contém ciclo dirigido')
    return ordem


## Experimento 1 — Materiais e circulação de apoio

In [ ]:
materiais,veiculos=criar_logistica()
for origem,esperado in [('AGUA',210),('PRODUTO',206)]:
    custo,rota=dijkstra(materiais,origem,'TALHAO')
    assert custo==esperado
    print(origem,':',' -> '.join(rota),'|',custo,'m')
ordem=ordem_topologica(materiais)
pos={v:i for i,v in enumerate(ordem)}
assert all(pos[a['origem']]<pos[a['destino']] for a in materiais.arestas.values())
print('Ordem topológica:',ordem)
custo,rota=dijkstra(veiculos,'BASE','PORTAO')
assert custo==65 and rota==['BASE','MANUTENCAO','PORTAO']
print('Veículo de apoio:',rota,'|',custo,'m')
try: ordem_topologica(veiculos)
except ValueError: print('Anel terrestre contém ciclo, como esperado.')
else: raise AssertionError('Anel não pode ser DAG')


AGUA : AGUA -> PREPARO -> ESTOQUE_CALDA -> ABASTECIMENTO -> DRONE -> TALHAO | 210.0 m
PRODUTO : PRODUTO -> PREPARO -> ESTOQUE_CALDA -> ABASTECIMENTO -> DRONE -> TALHAO | 206.0 m
Ordem topológica: ['AGUA', 'PRODUTO', 'PREPARO', 'ESTOQUE_CALDA', 'ABASTECIMENTO', 'DRONE', 'TALHAO']
Veículo de apoio: ['BASE', 'MANUTENCAO', 'PORTAO'] | 65.0 m
Anel terrestre contém ciclo, como esperado.


## Experimento 2 — Abastecimento, estoque e rastreabilidade

In [ ]:
def abastecer(estoque_L,volume_drone_L,pedido_L,pousado,lote_liberado,rede):
    tem_rota=bool(dijkstra(rede,'ESTOQUE_CALDA','DRONE')[1])
    liberado=(tem_rota and pousado and lote_liberado and 0<=estoque_L<=500
              and estoque_L/500*100>=20 and 0<=volume_drone_L<=30
              and 0<pedido_L<=estoque_L and volume_drone_L+pedido_L<=30)
    if not liberado: return {'liberado':False,'estoque_L':estoque_L,'drone_L':volume_drone_L}
    return {'liberado':True,'estoque_L':estoque_L-pedido_L,'drone_L':volume_drone_L+pedido_L}
transferencia=abastecer(250,0,24,True,True,materiais)
assert transferencia=={'liberado':True,'estoque_L':226,'drone_L':24}
registro={'lote':'SIM_001','missao':'MISSAO_001','talhao':'T1',
          'abastecido_L':24,'aplicado_L':5,'remanescente_L':19,'area_ha':0.5,'consumo_L_ha':10}
assert registro['abastecido_L']==registro['aplicado_L']+registro['remanescente_L']
assert registro['consumo_L_ha']==registro['aplicado_L']/registro['area_ha']
print('Transferência:',transferencia)
print('Registro de aplicação:',registro)
assert not abastecer(250,10,24,True,True,materiais)['liberado']
assert not abastecer(250,0,24,False,True,materiais)['liberado']
assert not abastecer(250,0,24,True,False,materiais)['liberado']
materiais.bloquear('m3')
assert dijkstra(materiais,'AGUA','TALHAO')==(inf,[])
assert not abastecer(250,0,24,True,True,materiais)['liberado']
print('Bloqueios de capacidade, estado, lote e caminho verificados.')


Transferência: {'liberado': True, 'estoque_L': 226, 'drone_L': 24}
Registro de aplicação: {'lote': 'SIM_001', 'missao': 'MISSAO_001', 'talhao': 'T1', 'abastecido_L': 24, 'aplicado_L': 5, 'remanescente_L': 19, 'area_ha': 0.5, 'consumo_L_ha': 10}
Bloqueios de capacidade, estado, lote e caminho verificados.
